# Pipeline Cheat Sheet — What to Call, and Why

**This notebook has no solved code on purpose.** It's a quick-reference companion for
`01_dataset_exploration.ipynb`, `02_preprocessing.ipynb`, and
`03_feature_engineering.ipynb` — a condensed map of *which function you reach for at
each point, and why*, without the full working logic already spelled out there. It goes
all the way through to the final step of Stage 4 (saving the engineered dataset) —
there's no shortened ending here.

Use it as a study sheet: cover the "why," try to say it in your own words before
reading it, then check yourself. If a line here doesn't make sense, that's a sign to go
back to the real notebook and actually run that step again, rather than move on.

See `FOLLOW_OR_DEFER.md` for what stage you should actually be working on right now.

## Stage 2 — Dataset Exploration

*(full working version: `01_dataset_exploration.ipynb`)*

In [ ]:
# CALL: pd.read_csv(filename)
# WHY:  loads the CSV into a DataFrame — nothing can happen before this.

# CALL: df.columns.str.strip() -> reassign to df.columns
# WHY:  CICIDS2017 files often have stray leading/trailing spaces in column
#       names (e.g. " Label"). Left unfixed, df["Label"] raises a KeyError
#       even though you can visibly see a "Label" column.

# CALL: df.shape
# WHY:  the very first sanity check on any new dataset — rows and columns.
#       If this looks wildly wrong (e.g. 60ish columns instead of ~79, or
#       columns that are numbers instead of names), STOP — something is
#       wrong with how the file loaded. Don't proceed past a wrong shape.

# CALL: df.head()
# WHY:  see real values, not just column names — catches obviously broken
#       loads immediately (e.g. columns that are actually data, see below).

# CALL: df.info()
# WHY:  shows data type per column (int64/float64/object). A numeric column
#       loaded as "object" (text) usually means hidden non-numeric junk in it.

# CALL: df.describe()
# WHY:  min/max/mean per numeric column. A max of `inf` here is your early
#       warning that some rate-based column (e.g. Flow Bytes/s) divided by
#       zero somewhere.

# CALL: df.isnull().sum()
# WHY:  counts missing values per column. Does NOT catch inf/-inf — that
#       needs a separate check (next line).

# CALL: np.isinf(df.select_dtypes(include=np.number)).sum()
# WHY:  isnull() misses infinite values entirely — this is the only way to
#       actually see them.

# CALL: df.duplicated().sum()
# WHY:  fully identical rows usually mean the same event was captured twice
#       — they'd bias training if kept (the model effectively sees that
#       example more than once).

# CALL: df["Label"].value_counts()  and  .value_counts(normalize=True)
# WHY:  shows every class and how common it is. This is where you discover
#       class imbalance (BENIGN usually dominates) — matters hugely for how
#       you'll read accuracy/F1 later. NEVER assume the label column name
#       or the class names — always read them from your actual output.

## Stage 3 — Preprocessing

*(full working version: `02_preprocessing.ipynb`)*

In [ ]:
# ORDER MATTERS. The single rule that governs this whole stage:
#   basic cleaning -> SPLIT -> fit any "learned" preprocessing on train only
# Anything that computes a statistic FROM the data (a mean, a scaler, an
# encoding, a "most important features" ranking) must happen AFTER the
# split, fit on X_train only. Row filters that don't use any such statistic
# (dropping a row because a value is inf, or NaN, or a duplicate) are safe
# to do BEFORE the split.

# CALL: np.isinf(df[numeric_cols]).any(axis=1)  -> boolean mask -> df[~mask]
# WHY:  removes rows with an undefined (infinite) rate value. This is a raw
#       row filter, not a learned statistic, so it's safe pre-split.

# CALL: df.dropna()
# WHY:  same reasoning — filtering on the raw presence of NaN, not a
#       statistic computed from the dataset's distribution.

# CALL: df.drop_duplicates()
# WHY:  same reasoning again — a row either duplicates another or it
#       doesn't; nothing is "learned."

# CALL: df.drop(columns=[...])  for Flow ID / Source IP / Destination IP / Timestamp
# WHY:  these are identifiers, not behaviour. Keeping them risks the model
#       memorizing specific IPs/flow IDs seen in training instead of
#       learning general attack patterns (data leakage / overfitting risk).
#       Ask: "would this value even exist for a brand-new real flow?"

# CALL: X = df.drop(columns=[label_col]); y = df[label_col]
# WHY:  separates what the model is allowed to see (X) from what it's
#       trying to predict (y).

# CALL: train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
# WHY:  THE split. stratify=y keeps class proportions similar in both
#       halves — important because rare attack classes could otherwise end
#       up almost entirely in one side by random chance. random_state=42
#       makes the split reproducible (same split every time you re-run it).

# CALL (pattern for ANY future scaler/encoder, not yet done in this project):
#       scaler.fit(X_train) -> scaler.transform(X_train) -> scaler.transform(X_test)
# WHY:  fit() learns statistics (e.g. mean/std) — doing that on X_test, or
#       on the full dataset before splitting, leaks test information into
#       training. transform() alone applies already-learned statistics —
#       safe to use on test data.

# CALL: X_train.to_csv(...); X_test.to_csv(...); y_train.to_csv(...); y_test.to_csv(...)
# WHY:  the LAST step of this stage — hands off a clean, split, leakage-safe
#       dataset to Stage 4 so it doesn't have to redo any of the cleaning.

## Stage 4 — Feature Engineering

*(guide: `03_FEATURE_ENGINEERING_GUIDE.md`, skeleton: `03_feature_engineering.ipynb`)*

Same call-and-why treatment as the stages above, all the way to the final step. This is
still yours to actually reason through and write — this is a map of *what* you're
reaching for at each point, not the reasoning itself.

In [ ]:
# STEP 1 — Load
# CALL: pd.read_csv(...) x4 for X_train, X_test, y_train, y_test
# WHY:  picks up exactly where Stage 3 left off — cleaned, split, ready.

# STEP 2 — Group features
# CALL: list(X_train.columns), then sort by hand into a dict, e.g.
#       {"traffic_volume": [...], "connection_behavior": [...], ...}
# WHY:  reasoning about 5-6 groups is far more tractable than reasoning
#       about 70+ individual columns one at a time — makes Step 4 possible.

# STEP 3 — Consider new features (optional, but must be a deliberate call)
# CALL: a new column derived from existing ones, e.g. df["a"] / df["b"],
#       applied identically to X_train and X_test
# WHY:  a ratio or derived value can sometimes capture a pattern the raw
#       columns don't directly express. "We decided not to add anything"
#       is also a valid, documented outcome here — this step isn't
#       mandatory to produce a new column, only to make and record the
#       decision either way.

# STEP 4 — Define feature-selection experiment sets
# CALL: sklearn.feature_selection tools (e.g. mutual_info_classif) fit on
#       X_train/y_train ONLY, then build 4 plain Python lists of column
#       names: experiment_A_features (large), _B (~50%), _C (most
#       important), _D (very small)
# WHY:  this is the actual research artifact of Stage 4 — concrete,
#       reusable feature sets that Stage 5 (not built yet) can train
#       against and compare, instead of picking a feature count by feel.
#       Same leakage rule as Stage 3: never fit a selection method on
#       X_test.

# STEP 5 — Finish the documentation table
# CALL: (no code — this is filling in research/feature_documentation_template.md)
# WHY:  every surviving column needs a recorded Meaning / Data Type /
#       Keep-Remove / Reason — the evidence that Stage 4 was reasoned
#       through, not guessed.

# STEP 6 — Save (the LAST step of Stage 4)
# CALL: X_train.to_csv("X_train_engineered.csv", ...); same for X_test;
#       plus the 4 feature-set lists saved as JSON/pickle/plain markdown
# WHY:  hands off the finished, engineered dataset AND the defined
#       experiment sets to Stage 5 — which is intentionally not started in
#       this repo yet (see FOLLOW_OR_DEFER.md). Do not train a model here.

---

**Remember:** this sheet is for review, not for copy-pasting into a real run. Actual
execution — with your actual data, actual column names, actual outputs — happens in
`01_dataset_exploration.ipynb`, `02_preprocessing.ipynb`, and
`03_feature_engineering.ipynb`. Once every step above is done for real, check it off in
`CHECKLIST.md`.